In [1]:
import numpy as np
from numpy.typing import NDArray

In [2]:
BOARD_LENGTH = 5
board = np.zeros((BOARD_LENGTH,BOARD_LENGTH), dtype=np.int8)
answer_dict = {} # Answer_dict maps the number of slots (k) until bingo => (# of possible ways a bingo could occur in k tiles, number of ways to "legitimately" fill a board with k tiles)
for i in range(BOARD_LENGTH, 22):
    answer_dict[i] = 0

def IsBingo(board: NDArray[np.int8]) -> bool:
    if BOARD_LENGTH in np.sum(board, axis=0):
        return True
    if BOARD_LENGTH in np.sum(board, axis=1):
        return True
    if np.sum(np.diag(board)) == BOARD_LENGTH:
        return True
    if np.sum(np.diag(np.fliplr(board))) == BOARD_LENGTH:
        return True
    return False

def GetNextIndex(currentIndex: tuple[int, int]) -> tuple[int, int]:
    if currentIndex[1] == BOARD_LENGTH - 1:
        return (currentIndex[0] + 1, 0)
    return (currentIndex[0], currentIndex[1] + 1)


def Recursive_Fill(currentIndex: tuple[int, int], depth: int):
    '''Returns the total number of Bingos accumulated in the first element, and the total number of board configurations accumulated in the second element'''
    total_slots = np.sum(board).astype(int)

    if IsBingo(board):
        answer_dict[total_slots] += 1
        return None
    
    while currentIndex[0] < BOARD_LENGTH and currentIndex[1] < BOARD_LENGTH:
        nextIndex = GetNextIndex(currentIndex)
        board[currentIndex] = 1
        Recursive_Fill(nextIndex, depth + 1)
        board[currentIndex] = 0
        currentIndex = nextIndex

    # At this point, the current layout is not a bingo, and we've reached the end of the board
    return None

In [3]:
Recursive_Fill((0,0), 0)
answer_dict

{5: 12,
 6: 176,
 7: 1430,
 8: 7590,
 9: 29025,
 10: 84198,
 11: 190739,
 12: 342388,
 13: 488718,
 14: 550959,
 15: 481868,
 16: 316707,
 17: 148594,
 18: 45873,
 19: 8162,
 20: 654,
 21: 13}

In [4]:
total_number_of_outcomes = np.sum([answer_dict[x] for x in answer_dict.keys()])
expected_number_of_tiles = np.sum([x* answer_dict[x] / total_number_of_outcomes for x in answer_dict.keys()])
expected_number_of_tiles

np.float64(13.81635389932765)

# Negative Hypergeometric Distribution

In [115]:
from math import comb

N = 10
S = 3
r = 2


def pmf(k: int) -> np.float64:
    return comb(r + k - 1, k) * comb(N-k-r, N-S-k) / comb(N, N-S)


In [144]:
probs = [answer_dict[x] / total_number_of_outcomes for x in answer_dict.keys()]

p_z = {}

for i,x in enumerate(answer_dict.keys()):
    p_z[x] = probs[i]
p_z

{5: np.float64(4.4492133420043555e-06),
 6: np.float64(6.525512901606389e-05),
 7: np.float64(0.0005301979232555191),
 8: np.float64(0.002814127438817755),
 9: np.float64(0.010761534770973035),
 10: np.float64(0.03121790541417356),
 11: np.float64(0.07071987530338074),
 12: np.float64(0.12694643814518228),
 13: np.float64(0.1812008871731404),
 14: np.float64(0.20427784447478148),
 15: np.float64(0.1786611278904129),
 16: np.float64(0.11742475082551446),
 17: np.float64(0.055093867278482936),
 18: np.float64(0.017008230303147152),
 19: np.float64(0.003026206608119963),
 20: np.float64(0.0002424821271392374),
 21: np.float64(4.819981120504719e-06)}

In [ ]:
N = 75
S = 25
X_low = 5
X_high = 71
Z_low = 5
Z_high = 21

E_X = (1 / comb(N, S)) * np.sum([k * (np.sum([comb(k-1, k-j) * comb(N - k, N - S - (k - j)) * p_z[j] for j in range(max(Z_low, k - 50), min(k, Z_high + 1))])) for k in range(X_low, X_high + 1)])
E_X

np.float64(40.38625854776962)